In [ ]:
#| hide
from fossick import *

# fossick

> web search, fetch, crawl, and browser automation for Python and AI agents

fossick covers the web-interaction stack that keeps coming up in Python and agent workflows: private local search, reading any page from a static blog to a JS-heavy SPA behind bot detection, and direct Chrome automation for authenticated sites, form filling, and multi-step flows.

The modules compose naturally. `search()` finds URLs; `fetch()` reads them — handling JavaScript rendering and anti-bot evasion automatically. For YouTube, arXiv papers, or GitHub repos, dedicated readers return structured data. When the page requires a real login session or interactive interaction, `cdp_connect()` attaches to a running Chrome browser: `pg.snapshot()` hands an agent a compact map of the page and `pg.fill_form()` / `pg.act()` drive it.

## Install

```sh
uv add fossick
```

Text/image/news search work out of the box (no Docker). JS rendering, `stealthy` fetching, and `google()` use a bundled headless browser.

## Search

`search()` runs a metasearch through [`ddgs`](https://github.com/deedy5/ddgs), aggregating many backends (Bing, Brave, DuckDuckGo, Startpage, Mojeek, Wikipedia, …) — no API key, no Docker. `images()`, `news()`, and `videos()` return the corresponding media.

For real Google ranking when you specifically need it, `google()` uses a stealth browser to bypass the JavaScript/bot wall that blocks plain-HTTP scraping. Every result is a plain dict, and all functions share a local TTL cache so repeated queries return instantly.

In [ ]:
results = search('fasthtml python web framework', n=5)
for r in results: print(r['title'], r['href'])

## Fetch and read

`fetch()` returns a page dict with the raw HTML, parsed JSON if the response was JSON, and any XHR calls the page made. `to_md()` converts HTML to clean markdown, optionally narrowing to a CSS selector first. Pass `heavy=True` for JavaScript-rendered pages or `stealthy=True` for sites with anti-bot detection.

fossick also pulls transcripts and metadata from YouTube, papers and PDFs from arXiv, and files from GitHub repos.

Pass `auto=True` to let `fetch()` escalate on its own — plain HTTP first, then a headless browser, then the stealth fetcher, then the logged-in debug Chrome — stopping at the first response that isn't a bot wall (the winning tier is on `page.tier`). Pass `session=True` to route the fetch through your persistent debug Chrome so it reuses that browser's logged-in cookies — read authenticated pages with no login code.

In [ ]:
# extract just the lead paragraphs — no nav, ads, or sidebars
page = fetch('https://en.wikipedia.org/wiki/Web_scraping', verify=False)
print(to_md(page, sel='.mw-parser-output > p')[:400])

[2026-06-22 20:32:21] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Web_scraping> (referer: https://www.google.com/)


`crawl()` follows links from a start URL, returning a list of Page dicts — useful for documentation sites, blogs, and any multi-page content.

In [ ]:
pages = crawl('https://docs.python.org/3/library/functions.html',
              follow_sel='a.reference.internal', same_domain=True, max_pages=3, verify=False)
print(f'{len(pages)} pages crawled')

[2026-06-22 20:32:23] INFO: Fetched (200) <GET https://docs.python.org/3/library/functions.html> (referer: https://www.google.com/)
[2026-06-22 20:32:23] INFO: Fetched (200) <GET https://docs.python.org/3/reference/datamodel.html#object.__abs__> (referer: https://www.google.com/)
[2026-06-22 20:32:23] INFO: Fetched (200) <GET https://docs.python.org/3/glossary.html#term-asynchronous-iterator> (referer: https://www.google.com/)


3 pages crawled


`fetch_all()` fetches multiple URLs in parallel — faster than sequential `fetch()` calls.

In [ ]:
urls = ['https://httpbin.org/get', 'https://httpbin.org/status/200', 'https://httpbin.org/json']
pages = fetch_all(urls, verify=False)
print([p.status for p in pages])

[2026-06-22 20:32:26] INFO: Fetched (503) <GET https://httpbin.org/json> (referer: https://www.google.com/)
[2026-06-22 20:32:26] INFO: Fetched (503) <GET https://httpbin.org/get> (referer: https://www.google.com/)
[2026-06-22 20:32:26] INFO: Fetched (503) <GET https://httpbin.org/status/200> (referer: https://www.google.com/)


[503, 503, 503]


`research()` closes the loop from question to answer: it searches, reads the top results in parallel (auto-escalating past bot walls), and returns one cited markdown corpus — `{query, sources, digest}` — ready to hand to an LLM.

In [ ]:
#| eval: false
notes = research('retrieval augmented generation best practices', n=5)
print(notes['digest'][:500])          # cited markdown, one ## section per source
print([s['href'] for s in notes['sources']])

`read_arxiv()` fetches paper metadata and converts the full PDF to markdown. Pass an arxiv ID, abstract URL, or PDF URL. The PDF is saved to `save_dir`; results are cached in-process.

In [ ]:
paper = read_arxiv('2306.14881', save_dir='.', verify=False)
print(paper['title'])
print()
print(paper['summary'][:400])

Modeling the molecular gas content and CO-to-H2 conversion factors in low-metallicity star-forming dwarf galaxies

Low-metallicity dwarf galaxies often show no or little CO emission, despite the intense star formation observed in local samples. Both simulations and resolved observations indicate that molecular gas in low-metallicity galaxies may reside in small dense clumps, surrounded by a substantial amount of more diffuse gas, not traced by CO. Constraining the relative importance of CO-bright versus CO-dar


`read_gh_repo()` clones or fetches a GitHub repo and returns `{path: content}` for matched files. `read_gh_file()` reads a single file from a GitHub blob URL.

In [ ]:
files = read_gh_repo('https://github.com/vedicreader/kosha', globs=('README*',))
for path, content in files.items():
    print(path.split('/')[-1], f'({len(content)} chars)')

README.md (164718 chars)


In [ ]:
txt = read_gh_file('https://github.com/vedicreader/kosha/blob/main/pyproject.toml')
print(txt[:300])

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "koshas"
dynamic = ["version"]
description = "kosha (कोश) — a treasury of your repo and environment context for coding agents. FTS5 + vector search + call graph, no LLMs required."
readme = "README.md"
requir


`search_yt()` searches YouTube and returns metadata for each result. `read_yt()` fetches the full English transcript and metadata for a known video URL.

In [ ]:
hits = search_yt('3blue1brown neural networks', n=3)
for h in hits: print(h['title'], '\n  ', h['url'])

But what is a neural network? | Deep learning chapter 1 
   https://www.youtube.com/watch?v=aircAruvnKk
Gradient descent, how neural networks learn | Deep Learning Chapter 2 
   https://www.youtube.com/watch?v=IHZwWFHWa-w
Backpropagation, intuitively | Deep Learning Chapter 3 
   https://www.youtube.com/watch?v=Ilg3gGewQ5U


In [ ]:
video = read_yt('https://www.youtube.com/watch?v=aircAruvnKk')
print(video['title'])
print(video['source'][:300])

But what is a neural network? | Deep learning chapter 1
[Music] This is a three. It's sloppily written and rendered at an extremely low resolution of 28x 28 pixels. But your brain has no trouble recognizing it as a three. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean this, this, and this are


In [ ]:
#| eval: false
path = download_yt('https://www.youtube.com/watch?v=aircAruvnKk',
                   format='audio', save_dir='.')
print(path)  # Path('But what is a neural network.mp3')

   But what is a neural network？ ｜ Deep learning chapter 1.mp3


`url2nb()` converts any URL — HTML page, arXiv paper, or PDF — to a Jupyter notebook. `pdf2nb()` converts a PDF (local path or URL) to a notebook where each page becomes a markdown cell with an empty code cell below for annotations.

In [ ]:
#| eval: false
nb = url2nb('https://squiddev.medium.com/continuing-continuations-cps-in-python-47bba90c8d1e', verify=False)
print(nb)   # Path('continuing-continuations-cps-in-python-47bba90c8d1e.ipynb')

[2026-06-22 20:37:55] INFO: Fetched (200) <GET https://squiddev.medium.com/continuing-continuations-cps-in-python-47bba90c8d1e> (referer: https://www.google.com/)


continuing-continuations-cps-in-python-47bba90c8d1.ipynb


In [ ]:
#| eval: false
nb = pdf2nb('https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf', 'sdt.ipynb', verify=False)
print(nb)   # Path('/path/to/sdt.ipynb')

[2026-06-22 20:37:57] INFO: Fetched (200) <GET https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf> (referer: https://www.google.com/)


/Users/71293/code/personal/orgs/fossick/nbs/sdt.ipynb


## Discover hidden APIs

Most modern sites serve their data through undocumented JSON APIs rather than HTML. `find_xhr()` visits a page with a headless browser and captures every network call the browser makes. `paginate_api()` replays the request across all pages and collects the results.

For sites behind a login, pass `find_xhr(url, session=True)` to capture the calls through your authenticated debug Chrome. Each result carries a `capture` you can hand to `replay_xhr()`, which re-issues the request as a fast plain-HTTP call — reusing the browser's cookies — turning any logged-in site's internal API into a data feed.

In [ ]:
#| eval: false
posts = paginate_api(
    'https://jsonplaceholder.typicode.com/posts',
    payload={'_page': 1, '_limit': 10},
    page_field='_page',
    size_field='_limit',
    method='GET',
)
print(f'{len(posts)} posts collected')
print(posts[0]['title'])

[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=1&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=2&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=3&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=4&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=5&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=6&_limit=10> (referer: https://www.google.com/)
[2026-05-28 16:37:16] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=7&_limit=10> (referer: https://www.googl

100 posts collected
sunt aut facere repellat provident occaecati excepturi optio reprehenderit


In [ ]:
#| eval: false
# on JavaScript-heavy sites, intercept the hidden API with a headless browser
# Woolworths uses a GraphQL endpoint — find_xhr captures whatever calls the page makes
apis = find_xhr('https://www.woolworths.com.au/shop/browse/fruit-veg', pattern='*woolworths.com.au/graphql*')
print(f'{len(apis)} GraphQL calls captured')
for a in apis:
    data = a.get('data', {}).get('data', a.get('data', {}))
    print(a['url'], list(data.keys()) if isinstance(data, dict) else type(data).__name__)

[2026-05-28 16:34:05] INFO: Fetched (200) <GET https://www.woolworths.com.au/shop/browse/fruit-veg> (referer: https://www.google.com/)
[2026-05-28 16:34:05] INFO: Fetched (200) <GET https://www.woolworths.com.au/apis/ui/settings> (referer: https://www.woolworths.com.au/shop/browse/fruit-veg)
[2026-05-28 16:34:05] INFO: Fetched (200) <GET https://www.woolworths.com.au/api/ui/v2/bootstrap> (referer: https://www.woolworths.com.au/shop/browse/fruit-veg)
[2026-05-28 16:34:05] INFO: Fetched (201) <GET https://www.woolworths.com.au/ZbnWGh/bzsy/4o7V/Z91V/IVtmZUtEk/z1D9kLGuuLOkth/GBAqNwE/CCVe/VHhbPWMB> (referer: https://www.woolworths.com.au/shop/browse/fruit-veg)
[2026-05-28 16:34:05] INFO: Fetched (200) <GET https://www.woolworths.com.au/auth/heartbeat> (referer: https://www.woolworths.com.au/shop/browse/fruit-veg)
[2026-05-28 16:34:05] INFO: Fetched (200) <GET https://www.woolworths.com.au/Shop/DynamicContent2Panel?scheduleKey=/shop/browse/fruit-veg-bottom> (referer: https://www.woolworths.c

2 GraphQL calls captured
https://www.woolworths.com.au/graphql ['products']
https://www.woolworths.com.au/graphql ['products']


## Browser automation

`cdp_connect()` attaches to a running Chrome browser over the DevTools Protocol. The browser uses a persistent debug profile so cookies and SSO sessions survive across runs. Log in once to any enterprise or authenticated site; every subsequent call reuses that session.

`pg.snapshot()` gives an LLM a compact, interactive-elements-only view of the page — one `[#id] role "name"` line per button, link, and field — far smaller than the full `ax_tree()`. Act on it with `pg.fill_form({label: value}, submit='Sign in')`, `pg.click_settle()`, or the CSS bridges `pg.click_sel()` / `pg.fill_sel()`. `pg.act([...])` runs a whole declarative flow (`goto` / `fill` / `click` / `select` / `read`) in one call, and `ax_diff(before, after)` shows exactly what an action changed. `pg.md()` pulls the live, post-JavaScript page straight into fossick's markdown pipeline. `syncy()` runs any async CDP call synchronously.

`snapshot()` re-reads the tree for you, so IDs are always fresh. When you need the raw tree, print the full `ax_tree()` output — never truncate — and re-read it after each navigation, since backend IDs change.

In [ ]:
#| eval: false
from fossick.cdp import cdp_connect, syncy

cdp = syncy(cdp_connect())
pg  = syncy(cdp.open_page('https://example.com/login'))

# Compact, agent-ready view — just the actionable elements
print(syncy(pg.snapshot()))

# Fill the form by label and submit — no manual node IDs, waits handled for you
print(syncy(pg.fill_form({'Email': 'user@example.com', 'Password': 'secret'}, submit='Sign in')))

# Or drive a whole flow declaratively, collecting what you read
out = syncy(pg.act([
    ('goto', 'https://example.com/search'),
    ('fill', 'Search', 'web scraping'),
    ('click', 'Search'),
    ('read', '.results')]))
print(out['.results'][:400])

## Setup — the persistent debug Chrome

`fetch(url, session=True)` and the CDP tools use a long-lived Chrome with remote debugging on port 9223 — you don't have to start it yourself. `fetch(session=True)`, `cdp_ws()`, and `cdp_connect()` launch one **headless** on first use, on a persistent profile (`~/.cache/fastcdp/cdp-chrome`), so cookies and SSO sessions survive across runs.

To log in by hand (Cloudflare / SSO / Anubis) you need a **headed** window. A running Chrome can't switch modes, so quit it and relaunch with `headless=False` — see *Managing the debug Chrome* in the [cdp docs](cdp.html). To keep one always-on at login, install the bundled service file for your OS (`chrome_debug.plist` for launchd, `chrome_debug.service` for systemd, `chrome_debug_task.xml` for Task Scheduler).

## CLI

fossick ships a `fossick` command for shell scripts and agent harnesses that don't have a Python kernel. All commands accept `--as_json` to return JSON instead of markdown.

```sh
# fetch a page as markdown (--auto escalates plain->heavy->stealthy->session; --session uses the logged-in Chrome)
fossick fetch https://en.wikipedia.org/wiki/Web_scraping --sel '.mw-parser-output > p'

# research: search, then read the top results into one cited markdown corpus
fossick research "retrieval augmented generation best practices" --n 5

# compact, agent-ready accessibility snapshot of a live page in the debug Chrome
fossick ax https://example.com

# web search
fossick search "fasthtml python framework" --n 5

# read an arxiv paper (summary only by default; --source for full text)
fossick read-arxiv 2306.14881

# YouTube transcript
fossick read-yt https://www.youtube.com/watch?v=aircAruvnKk

# search YouTube
fossick search-yt "3blue1brown neural networks" --n 3

# download YouTube audio or video
fossick download-yt https://www.youtube.com/watch?v=aircAruvnKk --format audio

# convert a URL, PDF, or arXiv paper to a Jupyter notebook
fossick url2nb https://arxiv.org/abs/2306.14881

# capture outgoing network requests fired by a page (uses real Chrome session)
fossick calls https://example.com --pattern '*api*' --as_json

# interactive screenshot capture — overlays a button in Chrome, ✓ Done to finish
fossick collect https://example.com --save_dir shots

# click elements to annotate them with AX role + selector; saves labeled screenshot
fossick annotate https://example.com --save_dir shots

# install SKILL.md to .agents/skills/fossick/ and .claude/skills/fossick/
fossick install
```


In [ ]:
#| eval: false
from fossick.cdp import cdp_setup, cdp_connect, syncy, _debug_running
# Start a persistent debug Chrome you can log into; fetch(url, session=True) reuses it afterwards.
if _debug_running(9223):                       # a headless instance may already be running
    cdp = syncy(cdp_connect(port=9223))
    try: syncy(cdp.quit())                     # quit() drops the socket -> ConnectionClosedOK
    except Exception: pass
syncy(cdp_setup(9223, headless=False))         # headed: log in by hand, then use session=True

## MCP server

`fossick-mcp` exposes the whole toolkit over the Model Context Protocol, so Claude Code, Claude Desktop, Codex, and any other MCP client can drive fossick directly — search, fetch, readers, hidden-API discovery, and the logged-in debug Chrome (`browse` / `page_*` tools).

``` sh
uv add 'fossick'        # or: pip install 'fossick'
```

**Claude Code**

``` sh
claude mcp add fossick -- uvx --from 'fossick' fossick-mcp
```

**Codex** (`~/.codex/config.toml`)

``` toml
[mcp_servers.fossick]
command = "uvx"
args = ["--from", "fossick", "fossick-mcp"]
```

**Claude Desktop** (`claude_desktop_config.json`)

``` json
{"mcpServers": {"fossick": {"command": "uvx", "args": ["--from", "fossick", "fossick-mcp"]}}}
```

The server speaks stdio by default (`fossick-mcp --http` for Streamable HTTP). Tools mirror the Python/CLI API — see the [mcp docs](https://vedicreader.github.io/fossick/mcp.html) for the full list.